# Petroleum Supply Analysis — Phase 2
## 1. Verify Raw Data Access
Check that the historical and incremental raw files are available.

In [0]:
from pathlib import Path

# Location of the validated Phase 1 raw data.
raw_base_path = Path(
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded/"
    "20261003T114452938244Z"
)

if not raw_base_path.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {raw_base_path}")

for load_type in ("full_load", "incremental_load"):
    load_path = raw_base_path / load_type
    json_files = sorted(load_path.glob("page_*.json"))

    if not json_files:
        raise FileNotFoundError(f"No raw JSON files found in: {load_path}")

    total_size_mb = sum(
        file.stat().st_size for file in json_files
    ) / 1_000_000

    print(
        f"{load_type}: "
        f"{len(json_files)} files | "
        f"{total_size_mb:.2f} MB"
    )

print("Raw data access verified.")

full_load: 136 files | 236.33 MB
incremental_load: 1 files | 0.99 MB
Raw data access verified.


## 2. Configure Pipeline Parameters
Select the raw data batch and load type. These parameters allow
historical batches to be reprocessed without changing the pipeline code.

In [0]:
from pathlib import Path

# Default values point to the validated Phase 1 batch.
dbutils.widgets.text(
    "raw_root",
    "/Volumes/workspace/default/petroleum_raw/phase1_expanded",
    "Raw Data Root"
)

dbutils.widgets.text(
    "batch_id",
    "20261003T114452938244Z",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "load_type",
    "full_load",
    ["full_load", "incremental_load"],
    "Load Type"
)

raw_root = dbutils.widgets.get("raw_root").strip()
batch_id = dbutils.widgets.get("batch_id").strip()
load_type = dbutils.widgets.get("load_type").strip()

if not raw_root.startswith("/Volumes/"):
    raise ValueError("Raw Data Root must be a /Volumes/ path.")

if not batch_id or "/" in batch_id or batch_id in {".", ".."}:
    raise ValueError("Batch ID must be a single directory name.")

if load_type not in {"full_load", "incremental_load"}:
    raise ValueError("Unsupported load type.")

source_path = Path(raw_root) / batch_id / load_type
source_files = sorted(source_path.glob("page_*.json"))

if not source_files:
    raise FileNotFoundError(f"No raw JSON files found in: {source_path}")

print(f"Batch ID: {batch_id}")
print(f"Load type: {load_type}")
print(f"Source path: {source_path}")
print(f"Files available: {len(source_files)}")

Batch ID: 20261003T114452938244Z
Load type: full_load
Source path: /Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load
Files available: 136


##3. Define the Raw Data Schema
Define the expected observation fields explicitly. Preserve raw values as strings before validation and conversion in the Silver layer.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Preserve source field names and raw values.
observation_schema = StructType([
    StructField("period", StringType(), True),
    StructField("duoarea", StringType(), True),
    StructField("area-name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("product-name", StringType(), True),
    StructField("process", StringType(), True),
    StructField("process-name", StringType(), True),
    StructField("series", StringType(), True),
    StructField("series-description", StringType(), True),
    StructField("value", StringType(), True),
    StructField("units", StringType(), True),
])

print("Observation schema defined successfully.")
for field in observation_schema.fields:
    print(f"{field.name}: {field.dataType.simpleString()}")

Observation schema defined successfully.
period: string
duoarea: string
area-name: string
product: string
product-name: string
process: string
process-name: string
series: string
series-description: string
value: string
units: string


%md
## 4. Inspect the Raw JSON Structure
Inspect one raw file and one observation before defining the explicit Bronze schema.

In [0]:
import json

# Inspect one file from the selected batch.
sample_file = source_files[0]

with sample_file.open("r", encoding="utf-8") as file:
    payload = json.load(file)

print(f"Sample file: {sample_file.name}")
print(f"Top-level fields: {list(payload.keys())}")

# Support a response object or an API response wrapper.
response = payload.get("response", payload)
records = response.get("data", [])

if not records:
    raise ValueError("The selected file contains no data records.")

print(f"Records in this file: {len(records)}")
print("\nFirst observation:")
print(json.dumps(records[0], indent=2))

print("\nField names and JSON value types:")
for field_name, value in records[0].items():
    print(f"{field_name}: {type(value).__name__}")

Sample file: page_0001.json
Top-level fields: ['total', 'dateFormat', 'frequency', 'data']
Records in this file: 5000

First observation:
{
  "period": "2005-01",
  "duoarea": "R10-Z00",
  "area-name": "PADD 1",
  "product": "EPPA",
  "product-name": "Asphalt and Road Oil",
  "process": "EEX",
  "process-name": "Exports",
  "series": "MAPEXP11",
  "series-description": "East Coast (PADD 1) Exports of Asphalt and Road Oil (Thousand Barrels)",
  "value": "82",
  "units": "MBBL"
}

Field names and JSON value types:
period: str
duoarea: str
area-name: str
product: str
product-name: str
process: str
process-name: str
series: str
series-description: str
value: str
units: str


## 4. Define the Raw Data Schema
Define the expected observation fields explicitly. Preserve raw values
as strings before validation and conversion in the Silver layer.

Observation schema defined successfully.
period: string
duoarea: string
area-name: string
product: string
product-name: string
process: string
process-name: string
series: string
series-description: string
value: string
units: string


## 5. Test Reading a Raw File with the Explicit Schema
Read one JSON file and expand its data array into individual observations.

In [0]:
from pyspark.sql.types import ArrayType
from pyspark.sql import functions as F

# Observations are stored inside the top-level "data" array.
raw_file_schema = StructType([
    StructField("data", ArrayType(observation_schema), True),
])

# Test the explicit schema on one file.
sample_raw_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json(str(source_files[0]))
)

# Convert each observation in the array into a separate row.
sample_observations_df = (
    sample_raw_df
    .select(F.explode("data").alias("observation"))
    .select("observation.*")
)

print(f"Sample observation count: {sample_observations_df.count()}")
display(sample_observations_df.limit(5))

Sample observation count: 5000


period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
2005-01,R10-Z00,PADD 1,EPPA,Asphalt and Road Oil,EEX,Exports,MAPEXP11,East Coast (PADD 1) Exports of Asphalt and Road Oil (Thousand Barrels),82,MBBL
2005-01,R10-Z00,PADD 1,EPPA,Asphalt and Road Oil,EEX,Exports,MAPEXP12,East Coast (PADD 1) Exports of Asphalt and Road Oil (Thousand Barrels per Day),3,MBBL/D
2005-01,R20-Z00,PADD 2,EPPA,Asphalt and Road Oil,EEX,Exports,MAPEXP21,Midwest (PADD 2) Exports of Asphalt and Road Oil (Thousand Barrels),166,MBBL
2005-01,R20-Z00,PADD 2,EPPA,Asphalt and Road Oil,EEX,Exports,MAPEXP22,Midwest (PADD 2) Exports of Asphalt and Road Oil (Thousand Barrels per Day),5,MBBL/D
2005-01,R30-Z00,PADD 3,EPPA,Asphalt and Road Oil,EEX,Exports,MAPEXP31,Gulf Coast (PADD 3) Exports of Asphalt and Road Oil (Thousand Barrels),7,MBBL


## 6. Read the Selected Raw Batch
Read all JSON pages in the selected batch using the explicit schema
and count the extracted observations.

In [0]:
# Read every JSON page from the parameter-selected folder.
raw_batch_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json([str(file_path) for file_path in source_files])
)

# Expand the nested observations into rows.
raw_observations_df = (
    raw_batch_df
    .select(F.explode("data").alias("observation"))
    .select("observation.*")
)

observation_count = raw_observations_df.count()

print(f"Batch ID: {batch_id}")
print(f"Load type: {load_type}")
print(f"Input files: {len(source_files)}")
print(f"Observations read: {observation_count:,}")

Batch ID: 20261003T114452938244Z
Load type: full_load
Input files: 136
Observations read: 675,752


## 7. Validate Observation Keys
Check for missing observation keys and duplicate period-series
combinations within the selected batch.

In [0]:
# Each monthly observation is identified by period and series.
key_columns = ["period", "series"]

missing_key_condition = (
    F.col("period").isNull()
    | (F.trim(F.col("period")) == "")
    | F.col("series").isNull()
    | (F.trim(F.col("series")) == "")
)

missing_key_count = (
    raw_observations_df
    .filter(missing_key_condition)
    .count()
)

# Count repeated keys without removing any source records.
duplicate_keys_df = (
    raw_observations_df
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()

print(f"Records with missing keys: {missing_key_count:,}")
print(f"Duplicate key combinations: {duplicate_key_count:,}")

if missing_key_count == 0 and duplicate_key_count == 0:
    print("Observation key checks passed.")
else:
    print("Key issues detected. Review before proceeding.")
    display(duplicate_keys_df.limit(10))

Records with missing keys: 0
Duplicate key combinations: 0
Observation key checks passed.


## 8. Configure Pipeline Tables
Define the target tables for Bronze, Silver, quarantined records,
and pipeline execution logs.

In [0]:
# Keep all project tables in the existing workspace.default schema.
target_catalog = "workspace"
target_schema = "default"
table_prefix = f"{target_catalog}.{target_schema}"

bronze_table = f"{table_prefix}.petroleum_bronze"
silver_table = f"{table_prefix}.petroleum_silver"
quarantine_table = f"{table_prefix}.petroleum_quarantine"
execution_log_table = f"{table_prefix}.petroleum_execution_logs"

print(f"Bronze table: {bronze_table}")
print(f"Silver table: {silver_table}")
print(f"Quarantine table: {quarantine_table}")
print(f"Execution log table: {execution_log_table}")

Bronze table: workspace.default.petroleum_bronze
Silver table: workspace.default.petroleum_silver
Quarantine table: workspace.default.petroleum_quarantine
Execution log table: workspace.default.petroleum_execution_logs


## 9. Create the Execution Log Table
Create a Delta table to record processing results, execution times,
row counts, and errors for each pipeline operation.

In [0]:
# Use UTC consistently for pipeline timestamps.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Preserve existing logs when this setup cell is rerun.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {execution_log_table} (
        log_id STRING,
        run_id STRING,
        batch_id STRING,
        load_type STRING,
        layer STRING,
        source_reference STRING,
        target_table STRING,
        start_time TIMESTAMP,
        end_time TIMESTAMP,
        status STRING,
        rows_read BIGINT,
        rows_inserted BIGINT,
        rows_updated BIGINT,
        rows_quarantined BIGINT,
        error_message STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
""")

print(f"Execution log table ready: {execution_log_table}")

Execution log table ready: workspace.default.petroleum_execution_logs


## 10. Define the Execution Logging Function
Create a reusable function to save processing results for each
file or table operation.

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

# Reuse the explicitly defined schema of the execution log table.
execution_log_schema = spark.table(execution_log_table).schema


def write_execution_log(
    *,
    run_id,
    batch_id,
    load_type,
    layer,
    source_reference,
    target_table,
    start_time,
    end_time,
    status,
    rows_read=None,
    rows_inserted=None,
    rows_updated=None,
    rows_quarantined=None,
    error_message=None,
):
    """Append one operation result to the execution log table."""
    if status not in {"Success", "Failure"}:
        raise ValueError("Status must be Success or Failure.")

    log_record = {
        "log_id": str(uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "load_type": load_type,
        "layer": layer,
        "source_reference": str(source_reference),
        "target_table": target_table,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": rows_quarantined,
        "error_message": error_message,
        "load_timestamp": datetime.now(timezone.utc),
    }

    log_df = spark.createDataFrame(
        [log_record],
        schema=execution_log_schema,
    )

    log_df.write.format("delta").mode("append").saveAsTable(
        execution_log_table
    )


print("Execution logging function ready.")

Execution logging function ready.


## 11. Verify Execution Logging
Count observations in one sample file and record the validation
operation in the execution log table.

In [0]:
# Identify and time this validation operation.
validation_run_id = str(uuid4())
validation_start = datetime.now(timezone.utc)

validation_df = (
    spark.read
    .schema(raw_file_schema)
    .option("multiLine", "true")
    .json(str(source_files[0]))
    .select(F.explode("data").alias("observation"))
)

validation_count = validation_df.count()
validation_end = datetime.now(timezone.utc)

# This operation reads data without inserting into Bronze or Silver.
write_execution_log(
    run_id=validation_run_id,
    batch_id=batch_id,
    load_type=load_type,
    layer="Raw-validation",
    source_reference=source_files[0],
    target_table=None,
    start_time=validation_start,
    end_time=validation_end,
    status="Success",
    rows_read=validation_count,
    rows_inserted=0,
    rows_updated=0,
    rows_quarantined=0,
)

# Retrieve only the log entry for this validation run.
display(
    spark.table(execution_log_table)
    .filter(F.col("run_id") == validation_run_id)
    .select(
        "layer",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "start_time",
        "end_time",
    )
)

layer,status,rows_read,rows_inserted,rows_updated,start_time,end_time
Raw-validation,Success,5000,0,0,2026-10-10T09:07:06.013Z,2026-10-10T09:07:07.154Z


## 12. Create the Quarantine Table
Store rejected records with their source details and rejection
reasons for investigation and reprocessing.

In [0]:
# Preserve rejected records and their original payload for review.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table} (
        quarantine_id STRING,
        run_id STRING,
        batch_id STRING,
        load_type STRING,
        layer STRING,
        source_file STRING,
        record_index BIGINT,
        raw_payload STRING,
        rejection_reason STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
""")

print(f"Quarantine table ready: {quarantine_table}")

Quarantine table ready: workspace.default.petroleum_quarantine


## 13. Define the Raw Record Contract
Validate original observation fields and value types before parsing
them into the Bronze schema.

In [0]:
# The explicit observation schema defines the expected source fields.
expected_fields = set(observation_schema.fieldNames())
required_key_fields = {"period", "series"}


def validate_raw_record(record):
    """Return rejection reasons for a source observation."""
    if not isinstance(record, dict):
        return ["Observation must be a JSON object."]

    reasons = []
    actual_fields = set(record.keys())

    extra_fields = sorted(actual_fields - expected_fields)
    missing_fields = sorted(expected_fields - actual_fields)

    if extra_fields:
        reasons.append(
            f"Unexpected fields: {', '.join(extra_fields)}"
        )

    if missing_fields:
        reasons.append(
            f"Missing fields: {', '.join(missing_fields)}"
        )

    for field_name in sorted(actual_fields & expected_fields):
        value = record[field_name]

        # Null values are allowed here; key fields are checked below.
        if value is not None and not isinstance(value, str):
            reasons.append(
                f"Invalid type for {field_name}: "
                f"expected string, received {type(value).__name__}"
            )

        if field_name in required_key_fields:
            if value is None or (
                isinstance(value, str) and not value.strip()
            ):
                reasons.append(f"Empty observation key: {field_name}")

    return reasons


print("Raw record validation function ready.")

Raw record validation function ready.


## 14. Test Raw Record Validation
Use controlled in-memory examples to verify that the validator
detects schema changes and missing observation keys.

In [0]:
from copy import deepcopy

# Read one original observation without modifying the source file.
with source_files[0].open("r", encoding="utf-8") as file:
    test_payload = json.load(file)

test_response = test_payload.get("response", test_payload)
original_record = test_response["data"][0]

# Create independent copies for controlled validation tests.
extra_field_record = deepcopy(original_record)
extra_field_record["unexpected_column"] = "test"

changed_type_record = deepcopy(original_record)
changed_type_record["value"] = 82

missing_key_record = deepcopy(original_record)
missing_key_record.pop("series", None)

test_cases = [
    ("Original record", original_record, []),
    (
        "Extra column",
        extra_field_record,
        ["Unexpected fields: unexpected_column"],
    ),
    (
        "Changed value type",
        changed_type_record,
        ["Invalid type for value: expected string, received int"],
    ),
    (
        "Missing series key",
        missing_key_record,
        ["Missing fields: series"],
    ),
]

for test_name, test_record, expected_reasons in test_cases:
    actual_reasons = validate_raw_record(test_record)

    assert actual_reasons == expected_reasons, (
        f"{test_name} failed. Expected {expected_reasons}, "
        f"received {actual_reasons}"
    )

    print(f"PASS: {test_name}")
    print(f"  Validation reasons: {actual_reasons}")

print("\nAll raw record validation tests passed.")

PASS: Original record
  Validation reasons: []
PASS: Extra column
  Validation reasons: ['Unexpected fields: unexpected_column']
PASS: Changed value type
  Validation reasons: ['Invalid type for value: expected string, received int']
PASS: Missing series key
  Validation reasons: ['Missing fields: series']

All raw record validation tests passed.


## 15. Validate Records from a Raw File
Apply the raw record contract to a selected file and separate
accepted observations from rejected observations.

In [0]:
def validate_raw_file(file_path):
    """Validate one raw file and return accepted and rejected records."""
    file_path = Path(file_path)

    with file_path.open("r", encoding="utf-8") as source:
        payload = json.load(source)

    if not isinstance(payload, dict):
        raise ValueError("The file must contain a JSON object.")

    response = payload.get("response", payload)

    if not isinstance(response, dict):
        raise ValueError("The response must be a JSON object.")

    observations = response.get("data")

    if not isinstance(observations, list):
        raise ValueError("The data field must contain a JSON array.")

    accepted_records = []
    rejected_records = []

    for record_index, record in enumerate(observations):
        reasons = validate_raw_record(record)

        if reasons:
            rejected_records.append({
                "source_file": str(file_path),
                "record_index": record_index,
                "raw_payload": json.dumps(record, ensure_ascii=False),
                "rejection_reason": "; ".join(reasons),
            })
        else:
            accepted_records.append({
                "record_index": record_index,
                "observation": record,
            })

    return accepted_records, rejected_records


# Validate one real source file before integrating table writes.
accepted_records, rejected_records = validate_raw_file(source_files[0])

print(f"File: {source_files[0].name}")
print(f"Accepted records: {len(accepted_records):,}")
print(f"Rejected records: {len(rejected_records):,}")

for rejected_record in rejected_records[:3]:
    print(rejected_record["rejection_reason"])

File: page_0001.json
Accepted records: 5,000
Rejected records: 0


## 16. Define the Quarantine Writer
Save rejected records with their source details, rejection reason,
and processing metadata for investigation.

In [0]:
quarantine_schema = spark.table(quarantine_table).schema


def write_quarantine_records(
    rejected_records, *, run_id, batch_id, load_type, layer
):
    """Save rejected records for one processing attempt."""
    if not rejected_records:
        return 0

    processed_at = datetime.now(timezone.utc)

    quarantine_rows = [
        {
            "quarantine_id": str(uuid4()),
            "run_id": run_id,
            "batch_id": batch_id,
            "load_type": load_type,
            "layer": layer,
            "source_file": record["source_file"],
            "record_index": record["record_index"],
            "raw_payload": record["raw_payload"],
            "rejection_reason": record["rejection_reason"],
            "load_timestamp": processed_at,
        }
        for record in rejected_records
    ]

    quarantine_df = spark.createDataFrame(
        quarantine_rows,
        schema=quarantine_schema,
    )

    quarantine_df.write.format("delta").mode("append").saveAsTable(
        quarantine_table
    )

    return len(quarantine_rows)


print("Quarantine writer ready.")

Quarantine writer ready.


## 17. Test Schema Drift Quarantine
Create two controlled invalid records and verify that both are
saved to quarantine with their rejection reasons.

In [0]:
from copy import deepcopy

# Create test copies without changing the original source record.
valid_record = accepted_records[0]["observation"]

extra_column_record = deepcopy(valid_record)
extra_column_record["unexpected_column"] = "test"

changed_type_record = deepcopy(valid_record)
changed_type_record["value"] = 82

test_records = [extra_column_record, changed_type_record]
test_rejections = []

for record_index, record in enumerate(test_records):
    reasons = validate_raw_record(record)

    assert reasons, "The invalid test record was not rejected."

    test_rejections.append({
        "source_file": "controlled_test://schema_drift",
        "record_index": record_index,
        "raw_payload": json.dumps(record, ensure_ascii=False),
        "rejection_reason": "; ".join(reasons),
    })

test_run_id = str(uuid4())

saved_count = write_quarantine_records(
    test_rejections,
    run_id=test_run_id,
    batch_id="schema_drift_test",
    load_type=load_type,
    layer="Schema-drift-test",
)

# Verify records saved by this specific test run.
saved_test_df = (
    spark.table(quarantine_table)
    .filter(F.col("run_id") == test_run_id)
)

actual_count = saved_test_df.count()

assert saved_count == 2
assert actual_count == 2, "Expected two records in quarantine."

print(f"Rejected test records saved: {actual_count}")
print("Quarantine write test: PASS")

display(
    saved_test_df.select(
        "record_index",
        "rejection_reason",
        "source_file",
        "load_timestamp",
    ).orderBy("record_index")
)

Rejected test records saved: 2
Quarantine write test: PASS


record_index,rejection_reason,source_file,load_timestamp
0,Unexpected fields: unexpected_column,controlled_test://schema_drift,2026-10-10T09:10:14.597Z
1,"Invalid type for value: expected string, received int",controlled_test://schema_drift,2026-10-10T09:10:14.597Z


## 18. Define and Create the Bronze Table
Preserve source values as strings and include metadata for source
tracking, record identification, and ingestion time.

Each row represents an observation received from a source file.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
)

# Map source field names to consistent Bronze column names.
bronze_field_mapping = {
    "period": "period",
    "duoarea": "duoarea",
    "area-name": "area_name",
    "product": "product",
    "product-name": "product_name",
    "process": "process",
    "process-name": "process_name",
    "series": "series",
    "series-description": "series_description",
    "value": "value",
    "units": "units",
}

bronze_schema = StructType(
    [
        StructField(column_name, StringType(), True)
        for column_name in bronze_field_mapping.values()
    ]
    + [
        StructField("bronze_id", StringType(), False),
        StructField("record_hash", StringType(), False),
        StructField("source_system", StringType(), False),
        StructField("source_file", StringType(), False),
        StructField("record_index", LongType(), False),
        StructField("batch_id", StringType(), False),
        StructField("load_type", StringType(), False),
        StructField("run_id", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]
)

print(f"Bronze schema ready: {len(bronze_schema.fields)} columns.")

Bronze schema ready: 20 columns.


In [0]:
# Build the table definition from the explicit Bronze schema.
column_definitions = []

for field in bronze_schema.fields:
    sql_type = field.dataType.simpleString().upper()
    null_constraint = "" if field.nullable else " NOT NULL"

    column_definitions.append(
        f"`{field.name}` {sql_type}{null_constraint}"
    )

columns_sql = ",\n    ".join(column_definitions)

# Preserve any existing table and its data.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_table} (
    {columns_sql}
)
USING DELTA
""")

# Verify the table's column names and data types.
actual_schema = spark.table(bronze_table).schema

expected_columns = [
    (field.name, field.dataType.simpleString())
    for field in bronze_schema.fields
]

actual_columns = [
    (field.name, field.dataType.simpleString())
    for field in actual_schema.fields
]

if actual_columns != expected_columns:
    raise ValueError(
        "The existing Bronze table does not match the expected schema."
    )

print(f"Bronze table ready: {bronze_table}")
print("Bronze schema check: PASS")

spark.table(bronze_table).printSchema()

Bronze table ready: workspace.default.petroleum_bronze
Bronze schema check: PASS
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process_name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series_description: string (nullable = true)
 |-- value: string (nullable = true)
 |-- units: string (nullable = true)
 |-- bronze_id: string (nullable = false)
 |-- record_hash: string (nullable = false)
 |-- source_system: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- record_index: long (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- load_type: string (nullable = false)
 |-- run_id: string (nullable = false)
 |-- load_timestamp: timestamp (nullable = false)



## 19. Prepare Bronze Records with Metadata
Map accepted source fields to Bronze columns and attach source
details, stable record identifiers, and ingestion metadata.

In [0]:
import hashlib


def stable_hash(value):
    """Generate a deterministic SHA-256 hash from JSON-compatible data."""
    serialized = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    )
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def prepare_bronze_records(
    accepted_records,
    *,
    source_file,
    batch_id,
    load_type,
    run_id,
    load_timestamp,
):
    """Prepare validated observations using the Bronze data contract."""
    bronze_rows = []
    source_file = str(source_file)

    for accepted_record in accepted_records:
        observation = accepted_record["observation"]
        record_index = accepted_record["record_index"]
        record_hash = stable_hash(observation)

        # Exclude execution metadata so retries retain the same identity.
        bronze_id = stable_hash({
            "source_system": "EIA",
            "batch_id": batch_id,
            "load_type": load_type,
            "source_file": source_file,
            "record_index": record_index,
            "record_hash": record_hash,
        })

        bronze_row = {
            target_field: observation[source_field]
            for source_field, target_field in bronze_field_mapping.items()
        }

        bronze_row.update({
            "bronze_id": bronze_id,
            "record_hash": record_hash,
            "source_system": "EIA",
            "source_file": source_file,
            "record_index": record_index,
            "batch_id": batch_id,
            "load_type": load_type,
            "run_id": run_id,
            "load_timestamp": load_timestamp,
        })

        bronze_rows.append(bronze_row)

    return bronze_rows


print("Bronze record preparation function ready.")

Bronze record preparation function ready.


In [0]:
preview_file = source_files[0]
preview_accepted, preview_rejected = validate_raw_file(preview_file)

preview_run_id = str(uuid4())
preview_timestamp = datetime.now(timezone.utc)

preview_bronze_rows = prepare_bronze_records(
    preview_accepted,
    source_file=preview_file,
    batch_id=batch_id,
    load_type=load_type,
    run_id=preview_run_id,
    load_timestamp=preview_timestamp,
)

preview_bronze_df = spark.createDataFrame(
    preview_bronze_rows,
    schema=bronze_schema,
)

print(f"Prepared Bronze records: {len(preview_bronze_rows):,}")
print(f"Rejected records: {len(preview_rejected):,}")

display(
    preview_bronze_df.select(
        "period",
        "series",
        "value",
        "bronze_id",
        "batch_id",
        "source_file",
        "record_index",
        "load_timestamp",
    ).limit(5)
)

Prepared Bronze records: 5,000
Rejected records: 0


period,series,value,bronze_id,batch_id,source_file,record_index,load_timestamp
2005-01,MAPEXP11,82,fdb85c45c60ee53922154e88fe21bb222e9fa0a2f33dea58e5e2c73a814a9dc0,20261003T114452938244Z,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,0,2026-10-10T09:18:59.678Z
2005-01,MAPEXP12,3,344b8accc888c66fe79a48c3783ab976ac874d3a0bded9afe3f9c0354844bcbd,20261003T114452938244Z,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,1,2026-10-10T09:18:59.678Z
2005-01,MAPEXP21,166,46337a34913308db51e341b04c6df5509ffb966aecaae316470c6809f3887c26,20261003T114452938244Z,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,2,2026-10-10T09:18:59.678Z
2005-01,MAPEXP22,5,8b7037aa1d72d4baecacd601cd6fb5086415b443617d898f88d1bd621d28d770,20261003T114452938244Z,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,3,2026-10-10T09:18:59.678Z
2005-01,MAPEXP31,7,56b6f2772a68b4ceef0a887bb4a43b62c631ff01e87d0287d39c9ac0683d13ce,20261003T114452938244Z,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,4,2026-10-10T09:18:59.678Z


## 20. Ingest a Raw File into Bronze
Validate source records, quarantine invalid observations, insert
new Bronze records using MERGE, and log the execution outcome.

In [0]:
def ingest_raw_file_to_bronze(
    file_path, *, batch_id, load_type, bronze_table=bronze_table
):
    """Ingest one raw file and record its processing outcome."""
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    source_reference = str(file_path)
    view_name = f"bronze_source_{uuid4().hex}"

    rows_read = None
    rows_inserted = 0
    rows_quarantined = 0
    status = "Failure"
    error_message = None

    try:
        accepted, rejected = validate_raw_file(file_path)
        rows_read = len(accepted) + len(rejected)

        # Save invalid observations while allowing valid ones to proceed.
        rows_quarantined = None
        rows_quarantined = write_quarantine_records(
            rejected,
            run_id=run_id,
            batch_id=batch_id,
            load_type=load_type,
            layer="Raw-to-Bronze",
        )

        if accepted:
            bronze_rows = prepare_bronze_records(
                accepted,
                source_file=file_path,
                batch_id=batch_id,
                load_type=load_type,
                run_id=run_id,
                load_timestamp=datetime.now(timezone.utc),
            )

            source_df = spark.createDataFrame(
                bronze_rows,
                schema=bronze_schema,
            ).dropDuplicates(["bronze_id"])

            source_df.createOrReplaceTempView(view_name)

            # Preserve existing rows, including their original timestamps.
            rows_inserted = None

            spark.sql(f"""
                MERGE INTO {bronze_table} AS target
                USING {view_name} AS source
                ON target.bronze_id = source.bronze_id
                WHEN NOT MATCHED THEN INSERT *
            """).collect()

            # Only newly inserted rows carry this unique execution ID.
            rows_inserted = (
                spark.table(bronze_table)
                .filter(F.col("run_id") == run_id)
                .count()
            )

        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                layer="Raw-to-Bronze",
                source_reference=source_reference,
                target_table=bronze_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_inserted=rows_inserted,
                rows_updated=0,
                rows_quarantined=rows_quarantined,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": 0,
        "rows_quarantined": rows_quarantined,
    }


print("Raw-to-Bronze ingestion function ready.")

Raw-to-Bronze ingestion function ready.


In [0]:
first_file_result = ingest_raw_file_to_bronze(
    source_files[0],
    batch_id=batch_id,
    load_type=load_type,
)

for metric, value in first_file_result.items():
    print(f"{metric}: {value}")

# Display the execution log for this ingestion.
display(
    spark.table(execution_log_table)
    .filter(F.col("run_id") == first_file_result["run_id"])
    .select(
        "layer",
        "source_reference",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_time",
        "end_time",
    )
)

run_id: 952445ab-5170-47b9-888d-d5aaf8a12291
status: Success
rows_read: 5000
rows_inserted: 0
rows_updated: 0
rows_quarantined: 0


layer,source_reference,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
Raw-to-Bronze,/Volumes/workspace/default/petroleum_raw/phase1_expanded/20261003T114452938244Z/full_load/page_0001.json,Success,5000,0,0,0,2026-10-10T09:23:14.303Z,2026-10-10T09:23:18.710Z


## 21. Load a Raw Batch into Bronze
Process all source files in a parameterized folder, record each
file's outcome, and report any failures for retry.

In [0]:
def ingest_raw_batch_to_bronze(folder_path, *, batch_id, load_type):
    """Process a raw folder and report successful and failed files."""
    folder_path = Path(folder_path)

    if load_type not in {"full_load", "incremental_load"}:
        raise ValueError("Unsupported load type.")

    if not batch_id.strip():
        raise ValueError("Batch ID must not be empty.")

    if not folder_path.is_dir():
        raise FileNotFoundError(f"Source folder not found: {folder_path}")

    files = sorted(folder_path.glob("page_*.json"))

    if not files:
        raise FileNotFoundError(f"No source files found: {folder_path}")

    successful_results = []
    failed_files = []

    for file_number, file_path in enumerate(files, start=1):
        try:
            result = ingest_raw_file_to_bronze(
                file_path,
                batch_id=batch_id,
                load_type=load_type,
            )
            successful_results.append(result)

            print(
                f"[{file_number}/{len(files)}] {file_path.name}: "
                f"inserted={result['rows_inserted']:,}, "
                f"quarantined={result['rows_quarantined']:,}"
            )

        except Exception as exc:
            # Continue with other files; the file function attempts failure logging.
            failed_files.append({
                "source_file": str(file_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(
                f"[{file_number}/{len(files)}] "
                f"{file_path.name}: FAILED — {exc}"
            )

    summary = {
        "files_found": len(files),
        "files_succeeded": len(successful_results),
        "files_failed": len(failed_files),
        "rows_read": sum(r["rows_read"] for r in successful_results),
        "rows_inserted": sum(r["rows_inserted"] for r in successful_results),
        "rows_updated": sum(r["rows_updated"] for r in successful_results),
        "rows_quarantined": sum(
            r["rows_quarantined"] for r in successful_results
        ),
    }

    print("\nBatch summary")
    for metric, value in summary.items():
        print(f"{metric}: {value:,}")

    if failed_files:
        raise RuntimeError(
            f"{len(failed_files)} file(s) failed. "
            "Review the errors and execution logs before retrying. "
            "Summary row counts cover successful files only."
        )

    return summary


print("Raw batch ingestion function ready.")

Raw batch ingestion function ready.


In [0]:
batch_result = ingest_raw_batch_to_bronze(
    source_path,
    batch_id=batch_id,
    load_type=load_type,
)

[1/136] page_0001.json: inserted=0, quarantined=0
[2/136] page_0002.json: inserted=5,000, quarantined=0
[3/136] page_0003.json: inserted=5,000, quarantined=0
[4/136] page_0004.json: inserted=5,000, quarantined=0
[5/136] page_0005.json: inserted=5,000, quarantined=0
[6/136] page_0006.json: inserted=5,000, quarantined=0
[7/136] page_0007.json: inserted=5,000, quarantined=0
[8/136] page_0008.json: inserted=5,000, quarantined=0
[9/136] page_0009.json: inserted=5,000, quarantined=0
[10/136] page_0010.json: inserted=5,000, quarantined=0
[11/136] page_0011.json: inserted=5,000, quarantined=0
[12/136] page_0012.json: inserted=5,000, quarantined=0
[13/136] page_0013.json: inserted=5,000, quarantined=0
[14/136] page_0014.json: inserted=5,000, quarantined=0
[15/136] page_0015.json: inserted=5,000, quarantined=0
[16/136] page_0016.json: inserted=5,000, quarantined=0
[17/136] page_0017.json: inserted=5,000, quarantined=0
[18/136] page_0018.json: inserted=5,000, quarantined=0
[19/136] page_0019.json

## 22. Define and Create the Silver Table
Store validated monthly observations with proper data types.

Logical primary key: period and series.
The period date represents the first day of the observation month.
Bronze ingestion time and Silver processing time are stored separately.

In [0]:
from pyspark.sql.types import DateType, DoubleType

silver_schema = StructType([
    StructField("period", DateType(), False),
    StructField("duoarea", StringType(), True),
    StructField("area_name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("process", StringType(), True),
    StructField("process_name", StringType(), True),
    StructField("series", StringType(), False),
    StructField("series_description", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("units", StringType(), True),
    StructField("bronze_id", StringType(), False),
    StructField("record_hash", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("bronze_load_timestamp", TimestampType(), False),
    StructField("run_id", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
])

silver_key_columns = ["period", "series"]

print(f"Silver schema ready: {len(silver_schema.fields)} columns.")

Silver schema ready: 20 columns.


In [0]:
silver_column_definitions = []

for field in silver_schema.fields:
    sql_type = field.dataType.simpleString().upper()
    null_constraint = "" if field.nullable else " NOT NULL"

    silver_column_definitions.append(
        f"`{field.name}` {sql_type}{null_constraint}"
    )

silver_columns_sql = ",\n    ".join(silver_column_definitions)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    {silver_columns_sql}
)
USING DELTA
""")

expected_silver_columns = [
    (field.name, field.dataType.simpleString())
    for field in silver_schema.fields
]

actual_silver_columns = [
    (field.name, field.dataType.simpleString())
    for field in spark.table(silver_table).schema.fields
]

if actual_silver_columns != expected_silver_columns:
    raise ValueError(
        "The existing Silver table does not match the expected schema."
    )

print(f"Silver table ready: {silver_table}")
print("Silver schema check: PASS")

Silver table ready: workspace.default.petroleum_silver
Silver schema check: PASS


## 23. Clean and Validate Bronze Records for Silver
Convert monthly periods to dates and values to doubles.
Separate invalid records with rejection reasons while preserving
their original Bronze fields for quarantine.

In [0]:
def prepare_silver_dataframes(bronze_df, *, run_id, processed_at):
    """Return valid Silver rows and invalid Bronze rows with reasons."""
    checked_df = (
        bronze_df
        .withColumn("_period_text", F.trim(F.col("period")))
        .withColumn("_series_text", F.trim(F.col("series")))
        .withColumn(
            "_parsed_period",
            F.expr(
                "try_cast(concat(_period_text, '-01') AS DATE)"
            ),
        )
        .withColumn(
            "_parsed_value",
            F.expr("try_cast(trim(value) AS DOUBLE)"),
        )
    )

    invalid_period = (
        F.col("_period_text").isNull()
        | ~F.col("_period_text").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
        | F.col("_parsed_period").isNull()
    )

    invalid_series = (
        F.col("_series_text").isNull()
        | (F.col("_series_text") == "")
    )

    invalid_value = (
        F.col("value").isNotNull()
        & (
            F.col("_parsed_value").isNull()
            | F.isnan(F.col("_parsed_value"))
            | (F.abs(F.col("_parsed_value")) == F.lit(float("inf")))
        )
    )

    checked_df = checked_df.withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(invalid_period, F.lit("Invalid monthly period")),
            F.when(invalid_series, F.lit("Missing series identifier")),
            F.when(invalid_value, F.lit("Invalid numeric value")),
        ),
    )

    # Keep original Bronze values for investigating rejected records.
    rejected_df = (
        checked_df
        .filter(F.col("rejection_reason") != "")
        .select(*bronze_df.columns, "rejection_reason")
    )

    valid_df = checked_df.filter(F.col("rejection_reason") == "")

    text_columns = [
        "duoarea",
        "area_name",
        "product",
        "product_name",
        "process",
        "process_name",
        "series_description",
        "units",
    ]

    for column_name in text_columns:
        valid_df = valid_df.withColumn(
            column_name,
            F.trim(F.col(column_name)),
        )

    valid_df = (
        valid_df
        .withColumn("period", F.col("_parsed_period"))
        .withColumn("series", F.col("_series_text"))
        .withColumn("value", F.col("_parsed_value"))
        .withColumn(
            "bronze_load_timestamp",
            F.col("load_timestamp"),
        )
        .withColumn("run_id", F.lit(run_id))
        .withColumn(
            "load_timestamp",
            F.lit(processed_at).cast("timestamp"),
        )
    )

    # Hash cleaned business fields only; exclude execution metadata.
    business_columns = list(bronze_field_mapping.values())

    valid_df = valid_df.withColumn(
        "record_hash",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(name) for name in business_columns]),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )

    return (
        valid_df.select(*silver_schema.fieldNames()),
        rejected_df,
    )


print("Silver cleaning and validation function ready.")

Silver cleaning and validation function ready.


## 24. Handle Duplicate and Conflicting Observations
Keep one observation per period and series within the selected batch.
Collapse identical observations and separate conflicting versions
for quarantine before merging into Silver.

In [0]:
from pyspark.sql.window import Window


def resolve_silver_duplicates(valid_df, bronze_df):
    """Deduplicate valid rows and return conflicting source records."""
    key_columns = ["period", "series"]

    # Different business hashes for one key indicate conflicting values.
    conflicting_keys_df = (
        valid_df
        .groupBy(*key_columns)
        .agg(
            F.countDistinct("record_hash").alias("version_count")
        )
        .filter(F.col("version_count") > 1)
        .select(*key_columns)
    )

    conflicting_ids_df = (
        valid_df
        .join(conflicting_keys_df, key_columns, "inner")
        .select("bronze_id")
        .distinct()
    )

    # Preserve original Bronze fields for quarantine investigation.
    conflicting_records_df = (
        bronze_df
        .join(conflicting_ids_df, "bronze_id", "left_semi")
        .withColumn(
            "rejection_reason",
            F.lit(
                "Conflicting observations for the same period "
                "and series within the selected batch"
            ),
        )
    )

    non_conflicting_df = valid_df.join(
        conflicting_keys_df,
        key_columns,
        "left_anti",
    )

    # Select the same representative consistently on repeated runs.
    duplicate_window = (
        Window
        .partitionBy(*key_columns)
        .orderBy(
            F.col("bronze_load_timestamp").asc(),
            F.col("bronze_id").asc(),
        )
    )

    deduplicated_df = (
        non_conflicting_df
        .withColumn(
            "_duplicate_rank",
            F.row_number().over(duplicate_window),
        )
        .filter(F.col("_duplicate_rank") == 1)
        .drop("_duplicate_rank")
        .select(*silver_schema.fieldNames())
    )

    return deduplicated_df, conflicting_records_df


print("Silver duplicate handling function ready.")

Silver duplicate handling function ready.


## 25. Save Silver Rejections to Quarantine
Preserve rejected Bronze observations and their rejection reasons
with metadata identifying the Silver processing attempt.

In [0]:
def write_silver_quarantine(rejected_df, *, run_id):
    """Save Silver rejections and return the number written."""
    rejection_count = rejected_df.count()

    if rejection_count == 0:
        return 0

    # Preserve the original source field names and Bronze values.
    raw_payload = F.to_json(
        F.struct(*[
            F.col(bronze_field).alias(source_field)
            for source_field, bronze_field
            in bronze_field_mapping.items()
        ]),
        options={"ignoreNullFields": "false"},
    )

    quarantine_df = rejected_df.select(
        F.expr("uuid()").alias("quarantine_id"),
        F.lit(run_id).alias("run_id"),
        F.col("batch_id"),
        F.col("load_type"),
        F.lit("Bronze-to-Silver").alias("layer"),
        F.col("source_file"),
        F.col("record_index"),
        raw_payload.alias("raw_payload"),
        F.col("rejection_reason"),
        F.lit(datetime.now(timezone.utc))
        .cast("timestamp")
        .alias("load_timestamp"),
    )

    (
        quarantine_df
        .select(*quarantine_schema.fieldNames())
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(quarantine_table)
    )

    return rejection_count


print("Silver quarantine writer ready.")

Silver quarantine writer ready.


## 26. Process a Bronze Batch into Silver
Clean and validate a selected batch, quarantine rejected records,
and merge valid observations into Silver.

Existing rows are updated only when their cleaned business data changes.
Every execution records its outcome and processing metrics.

In [0]:
def process_bronze_to_silver(
    *,
    batch_id,
    load_type,
    bronze_table=bronze_table,
    silver_table=silver_table,
    backfill_mode=False,
):
    """Process a Bronze batch with auditing and optional insert-only backfill."""
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    view_name = f"silver_source_{uuid4().hex}"

    source_reference = (
        f"{bronze_table}; batch_id={batch_id}; load_type={load_type}; "
        f"backfill_mode={backfill_mode}"
    )

    rows_read = None
    rows_inserted = 0
    rows_updated = 0
    rows_quarantined = 0
    status = "Failure"
    error_message = None

    try:
        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        if not isinstance(backfill_mode, bool):
            raise ValueError("backfill_mode must be True or False.")

        # Select only the requested historical or incremental batch.
        bronze_batch_df = (
            spark.table(bronze_table)
            .filter(
                (F.col("batch_id") == batch_id)
                & (F.col("load_type") == load_type)
            )
        )

        rows_read = bronze_batch_df.count()

        if rows_read == 0:
            raise ValueError("No Bronze records match the selected batch.")

        valid_df, invalid_df = prepare_silver_dataframes(
            bronze_batch_df,
            run_id=run_id,
            processed_at=start_time,
        )

        merge_df, conflicting_df = resolve_silver_duplicates(
            valid_df,
            bronze_batch_df,
        )

        rejected_df = invalid_df.unionByName(conflicting_df)

        rows_quarantined = None
        rows_quarantined = write_silver_quarantine(
            rejected_df,
            run_id=run_id,
        )

        merge_df.createOrReplaceTempView(view_name)

        # Backfill mode preserves existing observations and inserts missing ones.
        # Normal mode also updates observations whose business values changed.
        update_clause = ""

        if not backfill_mode:
            update_clause = """
                WHEN MATCHED
                    AND NOT (target.record_hash <=> source.record_hash)
                THEN UPDATE SET *
            """

        # Counts remain unknown until MERGE returns its operation metrics.
        rows_inserted = None
        rows_updated = None

        merge_output = spark.sql(f"""
            MERGE INTO {silver_table} AS target
            USING {view_name} AS source
            ON target.period = source.period
               AND target.series = source.series

            {update_clause}

            WHEN NOT MATCHED THEN INSERT *
        """).collect()

        if len(merge_output) != 1:
            raise RuntimeError("MERGE did not return its operation metrics.")

        merge_metrics = merge_output[0].asDict()
        required_metrics = {"num_inserted_rows", "num_updated_rows"}

        if not required_metrics.issubset(merge_metrics):
            raise RuntimeError(
                "Unexpected MERGE metrics: "
                f"{sorted(merge_metrics.keys())}"
            )

        rows_inserted = int(merge_metrics["num_inserted_rows"])
        rows_updated = int(merge_metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                layer="Bronze-to-Silver",
                source_reference=source_reference,
                target_table=silver_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                rows_quarantined=rows_quarantined,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": rows_quarantined,
    }


print("Bronze-to-Silver pipeline function ready.")

Bronze-to-Silver pipeline function ready.


In [0]:
silver_result = process_bronze_to_silver(
    batch_id=batch_id,
    load_type=load_type,
)

for metric, value in silver_result.items():
    print(f"{metric}: {value}")

display(
    spark.table(execution_log_table)
    .filter(F.col("run_id") == silver_result["run_id"])
    .select(
        "layer",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_time",
        "end_time",
    )
)

run_id: c868c4ce-5c5a-4fe2-a3fb-0f6e494bc775
status: Success
rows_read: 675752
rows_inserted: 675752
rows_updated: 0
rows_quarantined: 0


layer,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
Bronze-to-Silver,Success,675752,675752,0,0,2026-10-10T09:56:59.834Z,2026-10-10T09:57:17.374Z


## 27. Verify Silver Idempotency
Reprocess the same batch and compare Silver before and after.
Verify zero inserts, zero updates, unique keys, and unchanged
records, including processing timestamps.

In [0]:
# Pin the comparison to a specific Delta version before the rerun.
before_version = (
    spark.sql(f"DESCRIBE HISTORY {silver_table}")
    .select("version")
    .orderBy(F.col("version").desc())
    .first()["version"]
)

before_df = (
    spark.read
    .option("versionAsOf", str(before_version))
    .table(silver_table)
    .select(*silver_schema.fieldNames())
)

before_count = before_df.count()

# Execute the same production pipeline on the same input batch.
rerun_result = process_bronze_to_silver(
    batch_id=batch_id,
    load_type=load_type,
)

after_version = (
    spark.sql(f"DESCRIBE HISTORY {silver_table}")
    .select("version")
    .orderBy(F.col("version").desc())
    .first()["version"]
)

after_df = (
    spark.read
    .option("versionAsOf", str(after_version))
    .table(silver_table)
    .select(*silver_schema.fieldNames())
)

after_count = after_df.count()

duplicate_key_count = (
    after_df
    .groupBy(*silver_key_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Compare every column, including timestamps and ingestion metadata.
removed_or_changed = before_df.exceptAll(after_df).limit(1).count()
added_or_changed = after_df.exceptAll(before_df).limit(1).count()

assert rerun_result["rows_inserted"] == 0, "Unexpected inserts."
assert rerun_result["rows_updated"] == 0, "Unexpected updates."
assert before_count == after_count, "Silver row count changed."
assert duplicate_key_count == 0, "Duplicate Silver keys found."
assert removed_or_changed == 0, "Existing Silver records changed."
assert added_or_changed == 0, "New or changed Silver records found."

print(f"Rows before: {before_count:,}")
print(f"Rows after: {after_count:,}")
print(f"Rows inserted: {rerun_result['rows_inserted']}")
print(f"Rows updated: {rerun_result['rows_updated']}")
print(f"Duplicate keys: {duplicate_key_count}")
print("All values and timestamps unchanged: PASS")
print("Silver idempotency test: PASS")

Rows before: 675,752
Rows after: 675,752
Rows inserted: 0
Rows updated: 0
Duplicate keys: 0
All values and timestamps unchanged: PASS
Silver idempotency test: PASS


## 28. Process the Incremental Load
Load the subsequent-month raw files into Bronze, then process
the incremental batch into Silver using the existing pipeline functions.

In [0]:
incremental_source_path = (
    Path(raw_root) / batch_id / "incremental_load"
)

incremental_bronze_result = ingest_raw_batch_to_bronze(
    incremental_source_path,
    batch_id=batch_id,
    load_type="incremental_load",
)

[1/1] page_0001.json: inserted=2,820, quarantined=0

Batch summary
files_found: 1
files_succeeded: 1
files_failed: 0
rows_read: 2,820
rows_inserted: 2,820
rows_updated: 0
rows_quarantined: 0


In [0]:
incremental_silver_result = process_bronze_to_silver(
    batch_id=batch_id,
    load_type="incremental_load",
)

for metric, value in incremental_silver_result.items():
    print(f"{metric}: {value}")

display(
    spark.table(execution_log_table)
    .filter(
        F.col("run_id") == incremental_silver_result["run_id"]
    )
    .select(
        "layer",
        "load_type",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_time",
        "end_time",
    )
)

run_id: c50f8624-ddbc-458b-a478-7691ac9d10c1
status: Success
rows_read: 2820
rows_inserted: 2820
rows_updated: 0
rows_quarantined: 0


layer,load_type,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time
Bronze-to-Silver,incremental_load,Success,2820,2820,0,0,2026-10-10T10:02:00.221Z,2026-10-10T10:02:07.859Z


## 29. Select an Existing Observation for Revision Checking
Select a December 2025 observation and retain its current value
and source identifiers for comparison with a fresh EIA response.

In [0]:
# Select an existing observation with a numeric value.
revision_candidates = (
    spark.table(silver_table)
    .filter(
        (F.col("period") == F.lit("2025-12-01").cast("date"))
        & F.col("value").isNotNull()
        & F.col("duoarea").isNotNull()
        & F.col("product").isNotNull()
        & F.col("process").isNotNull()
    )
    .orderBy("series")
    .select(
        "period",
        "series",
        "duoarea",
        "product",
        "process",
        "value",
        "units",
        "bronze_id",
    )
    .limit(1)
    .collect()
)

if not revision_candidates:
    raise ValueError("No suitable December 2025 observation found.")

revision_candidate = revision_candidates[0].asDict()

print("Observation selected for revision checking:")
for field in [
    "period", "series", "duoarea", "product", "process", "value", "units"
]:
    print(f"{field}: {revision_candidate[field]}")

Observation selected for revision checking:
period: 2025-12-01
series: MAPEXP11
duoarea: R10-Z00
product: EPPA
process: EEX
value: 30.0
units: MBBL


## 30. Check the Selected Observation for an EIA Revision
Fetch the same monthly observation from EIA and compare its current
value with the saved Silver value. This step does not modify tables.

In [0]:
import requests
import math
from getpass import getpass


def fetch_revision_observation(candidate):
    """Fetch the selected observation without exposing the API key."""
    api_key = getpass("Enter your EIA API key: ").strip()

    if not api_key:
        raise ValueError("An EIA API key is required.")

    month = candidate["period"].strftime("%Y-%m")

    params = {
        "api_key": api_key,
        "frequency": "monthly",
        "data[0]": "value",
        "facets[duoarea][]": candidate["duoarea"],
        "facets[product][]": candidate["product"],
        "facets[process][]": candidate["process"],
        "start": month,
        "end": month,
        "offset": 0,
        "length": 5000,
    }

    try:
        response = requests.get(
            "https://api.eia.gov/v2/petroleum/sum/snd/data/",
            params=params,
            timeout=60,
        )
    except requests.RequestException:
        raise RuntimeError(
            "The EIA request failed. Check connectivity and retry."
        ) from None
    finally:
        params.pop("api_key", None)
        api_key = None

    if response.status_code != 200:
        raise RuntimeError(
            f"EIA returned HTTP {response.status_code}. "
            "Check your API key or retry later."
        )

    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError("EIA returned an invalid JSON response.") from None

    observations = payload.get("response", {}).get("data")

    if not isinstance(observations, list):
        raise ValueError("EIA did not return an observation list.")

    matches = [
        record
        for record in observations
        if record.get("period") == month
        and record.get("series") == candidate["series"]
        and record.get("duoarea") == candidate["duoarea"]
        and record.get("product") == candidate["product"]
        and record.get("process") == candidate["process"]
        and record.get("units") == candidate["units"]
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one matching observation; found {len(matches)}."
        )

    return matches[0]


fresh_revision_record = fetch_revision_observation(revision_candidate)

try:
    fresh_value = float(fresh_revision_record["value"])
except (TypeError, ValueError):
    raise ValueError("The fresh EIA value is not numeric.") from None

if not math.isfinite(fresh_value):
    raise ValueError("The fresh EIA value is not finite.")

saved_value = revision_candidate["value"]
revision_found = fresh_value != saved_value

print(f"Series: {revision_candidate['series']}")
print(f"Period: {revision_candidate['period']:%Y-%m}")
print(f"Saved value: {saved_value}")
print(f"Fresh EIA value: {fresh_value}")
print(f"Units: {revision_candidate['units']}")
print(f"Revision found: {revision_found}")

Enter your EIA API key:  [REDACTED]

Series: MAPEXP11
Period: 2025-12
Saved value: 30.0
Fresh EIA value: 30.0
Units: MBBL
Revision found: False


## 31. Check Monthly Observations for Actual Revisions
Fetch December 2025 observations with pagination and compare matching
series against Silver. Preserve fetched records for subsequent ingestion.

In [0]:
def fetch_eia_month(month):
    """Fetch all observations for one month without exposing credentials."""
    api_key = getpass("Enter your EIA API key: ").strip()

    if not api_key:
        raise ValueError("An EIA API key is required.")

    records = []
    offset = 0
    page_size = 5000

    params = {
        "api_key": api_key,
        "frequency": "monthly",
        "data[0]": "value",
        "start": month,
        "end": month,
        "sort[0][column]": "series",
        "sort[0][direction]": "asc",
        "length": page_size,
    }

    try:
        while True:
            params["offset"] = offset

            try:
                response = requests.get(
                    "https://api.eia.gov/v2/petroleum/sum/snd/data/",
                    params=params,
                    timeout=60,
                )
            except requests.RequestException:
                raise RuntimeError(
                    "The EIA request failed. Retry the monthly fetch."
                ) from None

            if response.status_code != 200:
                raise RuntimeError(
                    f"EIA returned HTTP {response.status_code}."
                )

            try:
                payload = response.json()
            except ValueError:
                raise RuntimeError(
                    "EIA returned invalid JSON."
                ) from None

            response_data = payload.get("response", {})
            page_records = response_data.get("data")

            if not isinstance(page_records, list):
                raise ValueError("EIA did not return an observation list.")

            total = int(response_data["total"])

            if not page_records and offset < total:
                raise RuntimeError("EIA returned an incomplete result.")

            records.extend(page_records)
            offset += len(page_records)

            print(f"Fetched {offset:,} of {total:,} observations.")

            if offset >= total:
                break

    finally:
        params.pop("api_key", None)
        api_key = None

    if any(record.get("period") != month for record in records):
        raise ValueError("The response contains an unexpected month.")

    keys = [(record.get("period"), record.get("series")) for record in records]

    if len(keys) != len(set(keys)):
        raise ValueError("The monthly response contains duplicate keys.")

    return records


revision_check_month = "2025-12"
fresh_month_records = fetch_eia_month(revision_check_month)

Enter your EIA API key:  [REDACTED]

Fetched 3,918 of 3,918 observations.


In [0]:
saved_month_rows = (
    spark.table(silver_table)
    .filter(
        F.col("period")
        == F.lit(f"{revision_check_month}-01").cast("date")
    )
    .select("series", "duoarea", "product", "process", "units", "value")
    .collect()
)

saved_by_key = {
    (
        row["series"],
        row["duoarea"],
        row["product"],
        row["process"],
        row["units"],
    ): row["value"]
    for row in saved_month_rows
}

revision_matches = []
compared_count = 0
skipped_count = 0

for record in fresh_month_records:
    key = (
        record.get("series"),
        record.get("duoarea"),
        record.get("product"),
        record.get("process"),
        record.get("units"),
    )

    if key not in saved_by_key:
        continue

    old_value = saved_by_key[key]

    try:
        new_value = (
            None if record.get("value") is None
            else float(record["value"])
        )
    except (TypeError, ValueError):
        skipped_count += 1
        continue

    if new_value is not None and not math.isfinite(new_value):
        skipped_count += 1
        continue

    compared_count += 1

    if old_value != new_value:
        revision_matches.append({
            "series": record["series"],
            "period": record["period"],
            "saved_value": old_value,
            "fresh_value": new_value,
            "units": record["units"],
            "raw_record": record,
        })

print(f"Matching observations compared: {compared_count:,}")
print(f"Invalid fresh values skipped: {skipped_count:,}")
print(f"Revised observations found: {len(revision_matches):,}")

for match in revision_matches[:10]:
    print(
        f"{match['series']} | {match['period']} | "
        f"{match['saved_value']} -> {match['fresh_value']} "
        f"{match['units']}"
    )

Matching observations compared: 2,828
Invalid fresh values skipped: 0
Revised observations found: 0


## 32. Prepare Isolated Tables for a Controlled Revision Test
Use the production pipeline functions with separate test tables.
The revised value will be synthetic and is not an actual EIA revision.

In [0]:
# Use unique table names to keep each test demonstration isolated.
revision_test_id = uuid4().hex[:12]

revision_test_bronze = (
    f"{table_prefix}.petroleum_revision_test_bronze_{revision_test_id}"
)
revision_test_silver = (
    f"{table_prefix}.petroleum_revision_test_silver_{revision_test_id}"
)


def create_test_table(table_name, schema):
    """Create an empty Delta test table with an explicit schema."""
    definitions = []

    for field in schema.fields:
        sql_type = field.dataType.simpleString().upper()
        null_constraint = "" if field.nullable else " NOT NULL"
        definitions.append(
            f"`{field.name}` {sql_type}{null_constraint}"
        )

    definition_sql = ",\n    ".join(definitions)

    spark.sql(f"""
        CREATE TABLE {table_name} (
            {definition_sql}
        )
        USING DELTA
    """)


create_test_table(revision_test_bronze, bronze_schema)
create_test_table(revision_test_silver, silver_schema)

print(f"Test Bronze table: {revision_test_bronze}")
print(f"Test Silver table: {revision_test_silver}")
print("Controlled revision test tables ready.")

Test Bronze table: workspace.default.petroleum_revision_test_bronze_6d65a278db3c
Test Silver table: workspace.default.petroleum_revision_test_silver_6d65a278db3c
Controlled revision test tables ready.


## 33. Load the Controlled Test Baseline
Copy the selected original Bronze observation into an isolated
test input file and process it through the production functions.

In [0]:
# Retrieve the original source values using the selected Bronze ID.
baseline_rows = (
    spark.table(bronze_table)
    .filter(F.col("bronze_id") == revision_candidate["bronze_id"])
    .limit(2)
    .collect()
)

if len(baseline_rows) != 1:
    raise ValueError("Expected exactly one original Bronze observation.")

original_bronze_record = baseline_rows[0].asDict()

test_baseline_record = {
    source_field: original_bronze_record[bronze_field]
    for source_field, bronze_field in bronze_field_mapping.items()
}

if validate_raw_record(test_baseline_record):
    raise ValueError("The test baseline does not satisfy the raw contract.")

test_baseline_batch_id = f"controlled_baseline_{revision_test_id}"

revision_test_root = (
    Path(raw_root) / "controlled_tests" / revision_test_id
)

test_baseline_folder = revision_test_root / "baseline" / "full_load"
test_baseline_folder.mkdir(parents=True, exist_ok=True)

test_baseline_file = test_baseline_folder / "page_0001.json"

# This file contains a copy; the original raw files remain intact.
baseline_payload = {
    "test_description": "Controlled MERGE test baseline",
    "data": [test_baseline_record],
}

with test_baseline_file.open("w", encoding="utf-8") as destination:
    json.dump(baseline_payload, destination, indent=2, ensure_ascii=False)

print(f"Test baseline file: {test_baseline_file}")
print(f"Series: {test_baseline_record['series']}")
print(f"Period: {test_baseline_record['period']}")
print(f"Baseline value: {test_baseline_record['value']}")

Test baseline file: /Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/baseline/full_load/page_0001.json
Series: MAPEXP11
Period: 2025-12
Baseline value: 30


In [0]:
test_baseline_bronze_result = ingest_raw_file_to_bronze(
    test_baseline_file,
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
)

test_baseline_silver_result = process_bronze_to_silver(
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
    silver_table=revision_test_silver,
)

test_baseline_snapshot = (
    spark.table(revision_test_silver)
    .select("period", "series", "value", "load_timestamp")
    .collect()
)

assert len(test_baseline_snapshot) == 1, "Expected one test Silver row."
assert test_baseline_snapshot[0]["value"] == float(
    test_baseline_record["value"]
), "The baseline value does not match the source."

print(f"Bronze rows inserted: {test_baseline_bronze_result['rows_inserted']}")
print(f"Silver rows inserted: {test_baseline_silver_result['rows_inserted']}")
print(f"Silver rows updated: {test_baseline_silver_result['rows_updated']}")
print(f"Baseline value: {test_baseline_snapshot[0]['value']}")
print("Controlled baseline check: PASS")

Bronze rows inserted: 1
Silver rows inserted: 1
Silver rows updated: 0
Baseline value: 30.0
Controlled baseline check: PASS


## 34. Controlled Revision Update Test

In [0]:
from copy import deepcopy
from decimal import Decimal

# Create a synthetic revision without changing the original record.
test_revised_record = deepcopy(test_baseline_record)
test_revised_record["value"] = str(
    Decimal(test_baseline_record["value"]) + Decimal("1")
)

if validate_raw_record(test_revised_record):
    raise ValueError("The revised record does not satisfy the raw contract.")

test_revision_batch_id = f"controlled_revision_{revision_test_id}"
test_revision_folder = (
    revision_test_root / "revision" / "incremental_load"
)
test_revision_folder.mkdir(parents=True, exist_ok=True)

test_revision_file = test_revision_folder / "page_0001.json"

revision_payload = {
    "test_description": "Synthetic revision for controlled MERGE testing",
    "data": [test_revised_record],
}

with test_revision_file.open("w", encoding="utf-8") as destination:
    json.dump(
        revision_payload,
        destination,
        indent=2,
        ensure_ascii=False,
    )

print(f"Original value: {test_baseline_record['value']}")
print(f"Revised value: {test_revised_record['value']}")
print(f"Test revision file: {test_revision_file}")

Original value: 30
Revised value: 31
Test revision file: /Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/revision/incremental_load/page_0001.json


In [0]:
# Capture the existing test row before applying the revision.
before_revision_rows = spark.table(revision_test_silver).collect()

assert len(before_revision_rows) == 1, "Expected one baseline row."

before_revision = before_revision_rows[0].asDict()

assert before_revision["value"] == float(test_baseline_record["value"]), (
    "The test table is not at the baseline value. "
    "The revision may already have been applied."
)

# Ingest the revised observation into the isolated test Bronze table.
test_revision_bronze_result = ingest_raw_file_to_bronze(
    test_revision_file,
    batch_id=test_revision_batch_id,
    load_type="incremental_load",
    bronze_table=revision_test_bronze,
)

# Apply the revision through the existing Silver MERGE pipeline.
test_revision_silver_result = process_bronze_to_silver(
    batch_id=test_revision_batch_id,
    load_type="incremental_load",
    bronze_table=revision_test_bronze,
    silver_table=revision_test_silver,
)

after_revision_rows = spark.table(revision_test_silver).collect()

assert len(after_revision_rows) == 1, "The revision created an extra row."

after_revision = after_revision_rows[0].asDict()

assert test_revision_silver_result["rows_inserted"] == 0, (
    "Expected zero new Silver rows."
)
assert test_revision_silver_result["rows_updated"] == 1, (
    "Expected exactly one Silver update."
)
assert after_revision["period"] == before_revision["period"]
assert after_revision["series"] == before_revision["series"]
assert after_revision["value"] == float(test_revised_record["value"])
assert after_revision["load_timestamp"] > before_revision["load_timestamp"]

print(f"Before value: {before_revision['value']}")
print(f"After value: {after_revision['value']}")
print(f"Silver rows inserted: {test_revision_silver_result['rows_inserted']}")
print(f"Silver rows updated: {test_revision_silver_result['rows_updated']}")
print(f"Test Silver row count: {len(after_revision_rows)}")
print("Controlled revision update: PASS")

# Show the audit entry for this Silver run.
display(
    spark.table(execution_log_table)
    .filter(
        F.col("run_id") == test_revision_silver_result["run_id"]
    )
    .select(
        "target_table",
        "load_type",
        "status",
        "rows_inserted",
        "rows_updated",
        "start_time",
        "end_time",
    )
)

Before value: 30.0
After value: 31.0
Silver rows inserted: 0
Silver rows updated: 1
Test Silver row count: 1
Controlled revision update: PASS


target_table,load_type,status,rows_inserted,rows_updated,start_time,end_time
workspace.default.petroleum_revision_test_silver_6d65a278db3c,incremental_load,Success,0,1,2026-10-10T10:38:01.523Z,2026-10-10T10:38:09.384Z


## 35. Revised Batch Idempotency Test

In [0]:
# Capture the complete Silver record before rerunning the revised batch.
rerun_before_rows = spark.table(revision_test_silver).collect()

assert len(rerun_before_rows) == 1, "Expected exactly one revised row."

rerun_before = rerun_before_rows[0].asDict()

assert rerun_before["value"] == float(test_revised_record["value"]), (
    "Complete the controlled revision update before this test."
)

# Reprocess the same file with the same batch ID and test tables.
revision_bronze_rerun = ingest_raw_file_to_bronze(
    test_revision_file,
    batch_id=test_revision_batch_id,
    load_type="incremental_load",
    bronze_table=revision_test_bronze,
)

revision_silver_rerun = process_bronze_to_silver(
    batch_id=test_revision_batch_id,
    load_type="incremental_load",
    bronze_table=revision_test_bronze,
    silver_table=revision_test_silver,
)

rerun_after_rows = spark.table(revision_test_silver).collect()

assert len(rerun_after_rows) == 1, "The rerun changed the Silver row count."

rerun_after = rerun_after_rows[0].asDict()

# Neither layer should insert, update, or quarantine any records.
for layer, result in (
    ("Bronze", revision_bronze_rerun),
    ("Silver", revision_silver_rerun),
):
    assert result["status"] == "Success", f"{layer} rerun failed."
    assert result["rows_inserted"] == 0, f"{layer} inserted new rows."
    assert result["rows_updated"] == 0, f"{layer} updated existing rows."
    assert result["rows_quarantined"] == 0, (
        f"{layer} unexpectedly quarantined records."
    )

# Compare every Silver column, including timestamps and run metadata.
assert rerun_before == rerun_after, (
    "The rerun changed the existing Silver record."
)

print(f"Bronze rows inserted: {revision_bronze_rerun['rows_inserted']}")
print(f"Silver rows inserted: {revision_silver_rerun['rows_inserted']}")
print(f"Silver rows updated: {revision_silver_rerun['rows_updated']}")
print(f"Silver value: {rerun_after['value']}")
print(f"Test Silver row count: {len(rerun_after_rows)}")
print(f"Timestamp before: {rerun_before['load_timestamp']}")
print(f"Timestamp after: {rerun_after['load_timestamp']}")
print("Complete Silver record unchanged: PASS")
print("Revised batch idempotency: PASS")

Bronze rows inserted: 0
Silver rows inserted: 0
Silver rows updated: 0
Silver value: 31.0
Test Silver row count: 1
Timestamp before: 2026-10-10 10:38:01.523402
Timestamp after: 2026-10-10 10:38:01.523402
Complete Silver record unchanged: PASS
Revised batch idempotency: PASS


## 36. Mixed-Record Schema Drift Test

In [0]:
from copy import deepcopy
from uuid import uuid4
import json

drift_test_id = uuid4().hex[:12]
drift_batch_id = f"controlled_schema_drift_{drift_test_id}"

drift_bronze_table = (
    f"workspace.default.petroleum_drift_test_bronze_{drift_test_id}"
)
drift_silver_table = (
    f"workspace.default.petroleum_drift_test_silver_{drift_test_id}"
)

# Create isolated tables using the existing explicit schemas.
for table_name, schema in (
    (drift_bronze_table, bronze_schema),
    (drift_silver_table, silver_schema),
):
    column_definitions = [
        f"`{field.name}` {field.dataType.simpleString().upper()}"
        + ("" if field.nullable else " NOT NULL")
        for field in schema.fields
    ]
    columns_sql = ",\n".join(column_definitions)

    spark.sql(
        f"CREATE TABLE {table_name} ({columns_sql}) USING DELTA"
    )

# Prepare one valid record and two deliberate contract violations.
drift_valid_record = deepcopy(test_revised_record)

drift_extra_column_record = deepcopy(drift_valid_record)
drift_extra_column_record["unexpected_field"] = "controlled_test"

drift_wrong_type_record = deepcopy(drift_valid_record)
drift_wrong_type_record["value"] = 31

drift_folder = (
    revision_test_root.parent
    / f"schema_drift_{drift_test_id}"
    / "incremental_load"
)
drift_folder.mkdir(parents=True, exist_ok=True)
drift_file = drift_folder / "page_0001.json"

with drift_file.open("w", encoding="utf-8") as destination:
    json.dump(
        {
            "test_description": "Controlled mixed-record schema drift test",
            "data": [
                drift_valid_record,
                drift_extra_column_record,
                drift_wrong_type_record,
            ],
        },
        destination,
        indent=2,
        ensure_ascii=False,
    )

print(f"Test file: {drift_file}")
print("Input records: 3")
print("Expected accepted records: 1")
print("Expected quarantined records: 2")

Test file: /Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/schema_drift_4317c07f77c8/incremental_load/page_0001.json
Input records: 3
Expected accepted records: 1
Expected quarantined records: 2


In [0]:
# Process the mixed file through the existing pipeline.
drift_bronze_result = ingest_raw_file_to_bronze(
    drift_file,
    batch_id=drift_batch_id,
    load_type="incremental_load",
    bronze_table=drift_bronze_table,
)

drift_silver_result = process_bronze_to_silver(
    batch_id=drift_batch_id,
    load_type="incremental_load",
    bronze_table=drift_bronze_table,
    silver_table=drift_silver_table,
)

# Verify that valid data was processed successfully.
assert drift_bronze_result["status"] == "Success"
assert drift_bronze_result["rows_read"] == 3
assert drift_bronze_result["rows_inserted"] == 1
assert drift_bronze_result["rows_quarantined"] == 2

assert drift_silver_result["status"] == "Success"
assert drift_silver_result["rows_read"] == 1
assert drift_silver_result["rows_inserted"] == 1
assert drift_silver_result["rows_updated"] == 0
assert drift_silver_result["rows_quarantined"] == 0

assert spark.table(drift_bronze_table).count() == 1

drift_silver_rows = spark.table(drift_silver_table).collect()
assert len(drift_silver_rows) == 1
assert drift_silver_rows[0]["series"] == drift_valid_record["series"]
assert drift_silver_rows[0]["value"] == float(drift_valid_record["value"])

# Verify that both invalid records were actually saved to quarantine.
drift_rejections = (
    spark.table(quarantine_table)
    .filter(F.col("run_id") == drift_bronze_result["run_id"])
    .select("record_index", "rejection_reason")
    .orderBy("record_index")
    .collect()
)

assert len(drift_rejections) == 2

reasons_by_index = {
    row["record_index"]: row["rejection_reason"]
    for row in drift_rejections
}

assert "unexpected_field" in reasons_by_index.get(1, "")
assert "Invalid type for value" in reasons_by_index.get(2, "")

print(f"Input records: {drift_bronze_result['rows_read']}")
print(f"Bronze rows inserted: {drift_bronze_result['rows_inserted']}")
print(f"Records quarantined: {len(drift_rejections)}")
print(f"Silver rows inserted: {drift_silver_result['rows_inserted']}")

for row in drift_rejections:
    print(f"Record {row['record_index']}: {row['rejection_reason']}")

print("Mixed-record schema drift test: PASS")

Input records: 3
Bronze rows inserted: 1
Records quarantined: 2
Silver rows inserted: 1
Record 1: Unexpected fields: unexpected_field
Record 2: Invalid type for value: expected string, received int
Mixed-record schema drift test: PASS


## 37. Verify Failure Logging

In [0]:
import json
from uuid import uuid4

failure_test_id = uuid4().hex[:12]
failure_batch_id = f"controlled_failure_{failure_test_id}"

failure_folder = (
    revision_test_root.parent
    / failure_batch_id
    / "incremental_load"
)
failure_folder.mkdir(parents=True, exist_ok=True)
failure_file = failure_folder / "page_0001.json"

# Write deliberately incomplete JSON to trigger a parsing failure.
failure_file.write_text('{"data": [', encoding="utf-8")

bronze_count_before = spark.table(drift_bronze_table).count()
expected_failure_caught = False

try:
    ingest_raw_file_to_bronze(
        failure_file,
        batch_id=failure_batch_id,
        load_type="incremental_load",
        bronze_table=drift_bronze_table,
    )
except json.JSONDecodeError:
    expected_failure_caught = True
    print("Expected JSON parsing failure caught.")

assert expected_failure_caught, "The malformed file did not fail as expected."

# Find the audit entry for this unique test batch.
failure_log_rows = (
    spark.table(execution_log_table)
    .filter(
        (F.col("batch_id") == failure_batch_id)
        & (F.col("layer") == "Raw-to-Bronze")
    )
    .collect()
)

assert len(failure_log_rows) == 1, "Expected one failure log entry."

failure_log = failure_log_rows[0].asDict()

assert failure_log["status"] == "Failure"
assert "JSONDecodeError" in (failure_log["error_message"] or "")
assert failure_log["source_reference"] == str(failure_file)
assert failure_log["target_table"] == drift_bronze_table
assert failure_log["load_type"] == "incremental_load"
assert failure_log["start_time"] is not None
assert failure_log["end_time"] is not None
assert failure_log["end_time"] >= failure_log["start_time"]
assert failure_log["rows_inserted"] == 0
assert failure_log["rows_updated"] == 0

bronze_count_after = spark.table(drift_bronze_table).count()
assert bronze_count_after == bronze_count_before, (
    "The failed file changed the Bronze row count."
)

print(f"Logged status: {failure_log['status']}")
print(f"Error: {failure_log['error_message']}")
print(f"Start time: {failure_log['start_time']}")
print(f"End time: {failure_log['end_time']}")
print(f"Rows inserted: {failure_log['rows_inserted']}")
print(f"Rows updated: {failure_log['rows_updated']}")
print("Bronze row count unchanged: PASS")
print("Failure logging test: PASS")

Expected JSON parsing failure caught.
Logged status: Failure
Error: JSONDecodeError: Expecting value: line 1 column 11 (char 10)
Start time: 2026-10-10 20:52:17.682935
End time: 2026-10-10 20:52:17.724629
Rows inserted: 0
Rows updated: 0
Bronze row count unchanged: PASS
Failure logging test: PASS


## 38. Add Backfill Protection
The function in Step 26 now accepts `backfill_mode=True`.
In this mode, missing observations are inserted and existing
Silver records remain unchanged.

## 39. Verify Historical Backfill Protection

In [0]:
# Restore references to the original controlled baseline.
test_baseline_batch_id = f"controlled_baseline_{revision_test_id}"
test_baseline_file = (
    revision_test_root / "baseline" / "full_load" / "page_0001.json"
)

with test_baseline_file.open("r", encoding="utf-8") as source:
    backfill_source_record = json.load(source)["data"][0]

# Capture the complete revised Silver record before the backfill.
backfill_before_rows = spark.table(revision_test_silver).collect()
assert len(backfill_before_rows) == 1, "Expected one revised Silver row."

backfill_before = backfill_before_rows[0].asDict()

assert backfill_before["value"] == float(test_revised_record["value"])
assert float(backfill_source_record["value"]) != backfill_before["value"], (
    "The baseline must differ from the revised value for this test."
)

# Replay the historical raw file using its original batch ID.
backfill_bronze_result = ingest_raw_file_to_bronze(
    test_baseline_file,
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
)

# Backfill mode preserves existing Silver observations.
backfill_silver_result = process_bronze_to_silver(
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
    silver_table=revision_test_silver,
    backfill_mode=True,
)

backfill_after_rows = spark.table(revision_test_silver).collect()
assert len(backfill_after_rows) == 1, "Backfill changed the row count."

backfill_after = backfill_after_rows[0].asDict()

for layer, result in (
    ("Bronze", backfill_bronze_result),
    ("Silver", backfill_silver_result),
):
    assert result["status"] == "Success", f"{layer} backfill failed."
    assert result["rows_read"] == 1
    assert result["rows_inserted"] == 0
    assert result["rows_updated"] == 0
    assert result["rows_quarantined"] == 0

assert backfill_before == backfill_after, (
    "Backfill changed the revised Silver record or its metadata."
)

# Verify that the backfill mode is recorded in the audit log.
backfill_logs = (
    spark.table(execution_log_table)
    .filter(F.col("run_id") == backfill_silver_result["run_id"])
    .collect()
)

assert len(backfill_logs) == 1
assert backfill_logs[0]["status"] == "Success"
assert "backfill_mode=True" in backfill_logs[0]["source_reference"]

print(f"Historical file value: {backfill_source_record['value']}")
print(f"Silver value before: {backfill_before['value']}")
print(f"Silver value after: {backfill_after['value']}")
print(f"Bronze rows inserted: {backfill_bronze_result['rows_inserted']}")
print(f"Silver rows inserted: {backfill_silver_result['rows_inserted']}")
print(f"Silver rows updated: {backfill_silver_result['rows_updated']}")
print("Complete Silver record unchanged: PASS")
print("Backfill mode recorded in audit log: PASS")
print("Historical backfill protection: PASS")

Historical file value: 30
Silver value before: 31.0
Silver value after: 31.0
Bronze rows inserted: 0
Silver rows inserted: 0
Silver rows updated: 0
Complete Silver record unchanged: PASS
Backfill mode recorded in audit log: PASS
Historical backfill protection: PASS


## 40. Backfill Missing Observations from Bronze

In [0]:
# Create an empty Silver target for an isolated backfill test.
missing_backfill_id = uuid4().hex[:12]
missing_backfill_silver = (
    f"workspace.default.petroleum_backfill_test_silver_{missing_backfill_id}"
)

backfill_columns_sql = ",\n".join(
    f"`{field.name}` {field.dataType.simpleString().upper()}"
    + ("" if field.nullable else " NOT NULL")
    for field in silver_schema.fields
)

spark.sql(
    f"CREATE TABLE {missing_backfill_silver} "
    f"({backfill_columns_sql}) USING DELTA"
)

assert spark.table(missing_backfill_silver).count() == 0

# Start directly from the existing historical Bronze batch.
missing_backfill_result = process_bronze_to_silver(
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
    silver_table=missing_backfill_silver,
    backfill_mode=True,
)

inserted_backfill_rows = spark.table(missing_backfill_silver).collect()

assert missing_backfill_result["status"] == "Success"
assert missing_backfill_result["rows_read"] == 1
assert missing_backfill_result["rows_inserted"] == 1
assert missing_backfill_result["rows_updated"] == 0
assert missing_backfill_result["rows_quarantined"] == 0
assert len(inserted_backfill_rows) == 1

inserted_backfill_record = inserted_backfill_rows[0].asDict()

assert inserted_backfill_record["series"] == backfill_source_record["series"]
assert str(inserted_backfill_record["period"]) == (
    f"{backfill_source_record['period']}-01"
)
assert inserted_backfill_record["value"] == float(
    backfill_source_record["value"]
)

# Repeat the backfill and verify that the complete row stays unchanged.
missing_backfill_rerun = process_bronze_to_silver(
    batch_id=test_baseline_batch_id,
    load_type="full_load",
    bronze_table=revision_test_bronze,
    silver_table=missing_backfill_silver,
    backfill_mode=True,
)

repeated_backfill_rows = spark.table(missing_backfill_silver).collect()

assert missing_backfill_rerun["status"] == "Success"
assert missing_backfill_rerun["rows_inserted"] == 0
assert missing_backfill_rerun["rows_updated"] == 0
assert missing_backfill_rerun["rows_quarantined"] == 0
assert len(repeated_backfill_rows) == 1
assert repeated_backfill_rows[0].asDict() == inserted_backfill_record

print(f"First backfill inserts: {missing_backfill_result['rows_inserted']}")
print(f"First backfill updates: {missing_backfill_result['rows_updated']}")
print(f"Inserted historical value: {inserted_backfill_record['value']}")
print(f"Repeated backfill inserts: {missing_backfill_rerun['rows_inserted']}")
print(f"Repeated backfill updates: {missing_backfill_rerun['rows_updated']}")
print("Complete record unchanged on rerun: PASS")
print("Backfill from existing Bronze: PASS")

First backfill inserts: 1
First backfill updates: 0
Inserted historical value: 30.0
Repeated backfill inserts: 0
Repeated backfill updates: 0
Complete record unchanged on rerun: PASS
Backfill from existing Bronze: PASS


## 41. Review Pipeline Audit Evidence

In [0]:
# Review the completed tests from this session.
evidence_run_ids = [
    revision_bronze_rerun["run_id"],
    revision_silver_rerun["run_id"],
    drift_bronze_result["run_id"],
    drift_silver_result["run_id"],
    failure_log["run_id"],
    backfill_bronze_result["run_id"],
    backfill_silver_result["run_id"],
    missing_backfill_result["run_id"],
    missing_backfill_rerun["run_id"],
]

audit_evidence_df = (
    spark.table(execution_log_table)
    .filter(F.col("run_id").isin(evidence_run_ids))
)

audit_rows = audit_evidence_df.collect()

assert len(audit_rows) == len(evidence_run_ids), (
    "Expected exactly one audit entry per selected run."
)
assert {row["run_id"] for row in audit_rows} == set(evidence_run_ids)

for row in audit_rows:
    expected_status = (
        "Failure" if row["run_id"] == failure_log["run_id"] else "Success"
    )
    assert row["status"] == expected_status
    assert row["start_time"] is not None
    assert row["end_time"] is not None
    assert row["end_time"] >= row["start_time"]
    assert row["source_reference"]
    assert row["target_table"]
    assert row["rows_inserted"] is not None
    assert row["rows_updated"] is not None

display(
    audit_evidence_df
    .select(
        "layer",
        "batch_id",
        "load_type",
        "source_reference",
        "target_table",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_time",
        "end_time",
        "error_message",
    )
    .orderBy("start_time")
)

print(f"Verified audit entries: {len(audit_rows)}")
print("Pipeline audit evidence: PASS")

layer,batch_id,load_type,source_reference,target_table,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time,error_message
Raw-to-Bronze,controlled_revision_6d65a278db3c,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/revision/incremental_load/page_0001.json,workspace.default.petroleum_revision_test_bronze_6d65a278db3c,Success,1,0,0,0,2026-10-10T20:39:48.459Z,2026-10-10T20:39:57.556Z,null
Bronze-to-Silver,controlled_revision_6d65a278db3c,incremental_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_revision_6d65a278db3c; load_type=incremental_load,workspace.default.petroleum_revision_test_silver_6d65a278db3c,Success,1,0,0,0,2026-10-10T20:40:04.461Z,2026-10-10T20:40:20.602Z,null
Raw-to-Bronze,controlled_schema_drift_4317c07f77c8,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/schema_drift_4317c07f77c8/incremental_load/page_0001.json,workspace.default.petroleum_drift_test_bronze_4317c07f77c8,Success,3,1,0,2,2026-10-10T20:50:45.952Z,2026-10-10T20:50:51.842Z,null
Bronze-to-Silver,controlled_schema_drift_4317c07f77c8,incremental_load,workspace.default.petroleum_drift_test_bronze_4317c07f77c8; batch_id=controlled_schema_drift_4317c07f77c8; load_type=incremental_load,workspace.default.petroleum_drift_test_silver_4317c07f77c8,Success,1,1,0,0,2026-10-10T20:50:53.874Z,2026-10-10T20:51:03.302Z,null
Raw-to-Bronze,controlled_failure_1a858a0dda8e,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/controlled_failure_1a858a0dda8e/incremental_load/page_0001.json,workspace.default.petroleum_drift_test_bronze_4317c07f77c8,Failure,null,0,0,0,2026-10-10T20:52:17.682Z,2026-10-10T20:52:17.724Z,JSONDecodeError: Expecting value: line 1 column 11 (char 10)
Raw-to-Bronze,controlled_baseline_6d65a278db3c,full_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/baseline/full_load/page_0001.json,workspace.default.petroleum_revision_test_bronze_6d65a278db3c,Success,1,0,0,0,2026-10-10T21:00:53.968Z,2026-10-10T21:00:57.247Z,null
Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_revision_test_silver_6d65a278db3c,Success,1,0,0,0,2026-10-10T21:01:00.581Z,2026-10-10T21:01:08.807Z,null
Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_backfill_test_silver_9c15a129eec1,Success,1,1,0,0,2026-10-10T21:02:01.526Z,2026-10-10T21:02:08.953Z,null
Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_backfill_test_silver_9c15a129eec1,Success,1,0,0,0,2026-10-10T21:02:11.390Z,2026-10-10T21:02:19.608Z,null


Verified audit entries: 9
Pipeline audit evidence: PASS


In [0]:
# Retrieve the successful controlled revision update from saved logs.
revision_update_evidence = (
    spark.table(execution_log_table)
    .filter(
        (F.col("batch_id") == test_revision_batch_id)
        & (F.col("layer") == "Bronze-to-Silver")
        & (F.col("target_table") == revision_test_silver)
        & (F.col("status") == "Success")
        & (F.col("rows_updated") >= 1)
    )
)

assert revision_update_evidence.count() >= 1, (
    "The controlled revision update log was not found."
)

complete_audit_evidence = (
    audit_evidence_df
    .unionByName(revision_update_evidence)
    .dropDuplicates(["log_id"])
)

display(
    complete_audit_evidence
    .select(
        "run_id",
        "layer",
        "batch_id",
        "load_type",
        "source_reference",
        "target_table",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "start_time",
        "end_time",
        "error_message",
    )
    .orderBy("start_time")
)

print("Controlled revision update included: PASS")

run_id,layer,batch_id,load_type,source_reference,target_table,status,rows_read,rows_inserted,rows_updated,rows_quarantined,start_time,end_time,error_message
2dd75677-4c15-4f8e-a4e7-fc4e5a3db913,Bronze-to-Silver,controlled_revision_6d65a278db3c,incremental_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_revision_6d65a278db3c; load_type=incremental_load,workspace.default.petroleum_revision_test_silver_6d65a278db3c,Success,1,0,1,0,2026-10-10T10:38:01.523Z,2026-10-10T10:38:09.384Z,null
417a0373-c9c3-43b2-8dce-e140345c5c37,Raw-to-Bronze,controlled_revision_6d65a278db3c,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/revision/incremental_load/page_0001.json,workspace.default.petroleum_revision_test_bronze_6d65a278db3c,Success,1,0,0,0,2026-10-10T20:39:48.459Z,2026-10-10T20:39:57.556Z,null
28dac506-ede4-4c33-96ea-4ada378eed8a,Bronze-to-Silver,controlled_revision_6d65a278db3c,incremental_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_revision_6d65a278db3c; load_type=incremental_load,workspace.default.petroleum_revision_test_silver_6d65a278db3c,Success,1,0,0,0,2026-10-10T20:40:04.461Z,2026-10-10T20:40:20.602Z,null
634a9648-9eac-489f-b0c5-a094225bab67,Raw-to-Bronze,controlled_schema_drift_4317c07f77c8,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/schema_drift_4317c07f77c8/incremental_load/page_0001.json,workspace.default.petroleum_drift_test_bronze_4317c07f77c8,Success,3,1,0,2,2026-10-10T20:50:45.952Z,2026-10-10T20:50:51.842Z,null
78dc009c-3b26-405d-bd9d-3eb0db09ae02,Bronze-to-Silver,controlled_schema_drift_4317c07f77c8,incremental_load,workspace.default.petroleum_drift_test_bronze_4317c07f77c8; batch_id=controlled_schema_drift_4317c07f77c8; load_type=incremental_load,workspace.default.petroleum_drift_test_silver_4317c07f77c8,Success,1,1,0,0,2026-10-10T20:50:53.874Z,2026-10-10T20:51:03.302Z,null
8c6ec9a1-6062-4472-8e7b-30d45aafd9fe,Raw-to-Bronze,controlled_failure_1a858a0dda8e,incremental_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/controlled_failure_1a858a0dda8e/incremental_load/page_0001.json,workspace.default.petroleum_drift_test_bronze_4317c07f77c8,Failure,null,0,0,0,2026-10-10T20:52:17.682Z,2026-10-10T20:52:17.724Z,JSONDecodeError: Expecting value: line 1 column 11 (char 10)
ca38ccb2-39fa-4e93-9a90-c62a473c60a6,Raw-to-Bronze,controlled_baseline_6d65a278db3c,full_load,/Volumes/workspace/default/petroleum_raw/phase1_expanded/controlled_tests/6d65a278db3c/baseline/full_load/page_0001.json,workspace.default.petroleum_revision_test_bronze_6d65a278db3c,Success,1,0,0,0,2026-10-10T21:00:53.968Z,2026-10-10T21:00:57.247Z,null
63910ca5-34e4-460f-bb2c-31f61e7c18ec,Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_revision_test_silver_6d65a278db3c,Success,1,0,0,0,2026-10-10T21:01:00.581Z,2026-10-10T21:01:08.807Z,null
cccb49a4-d2ad-48fb-99ad-6fd339ee310d,Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_backfill_test_silver_9c15a129eec1,Success,1,1,0,0,2026-10-10T21:02:01.526Z,2026-10-10T21:02:08.953Z,null
6ba194c5-094c-4283-95ea-e313cc97150d,Bronze-to-Silver,controlled_baseline_6d65a278db3c,full_load,workspace.default.petroleum_revision_test_bronze_6d65a278db3c; batch_id=controlled_baseline_6d65a278db3c; load_type=full_load; backfill_mode=True,workspace.default.petroleum_backfill_test_silver_9c15a129eec1,Success,1,0,0,0,2026-10-10T21:02:11.390Z,2026-10-10T21:02:19.608Z,null


Controlled revision update included: PASS
